# Qwen 2.5 Fine-tuning (Standard HuggingFace Stack)

Fine-tunes **Qwen2.5-0.5B-Instruct** using **QLoRA** (4-bit quantization + LoRA adapters)  
via standard HuggingFace + PEFT + bitsandbytes — no Unsloth required.

### Instructions
1. `Runtime` → `Change runtime type` → **T4 GPU**
2. Upload your `train.jsonl` to the file explorer on the left
3. Run all cells in order

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers==4.51.3 datasets peft accelerate
!pip install -q bitsandbytes --prefer-binary
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 131.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.4 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Verify GPU & versions ─────────────────────────────────────────────
import torch

assert torch.cuda.is_available(), "No GPU found! Enable GPU runtime first."

import transformers, peft, trl, bitsandbytes
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.version.cuda}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")

PyTorch      : 2.10.0+cu128
CUDA         : 12.8
GPU          : Tesla T4
transformers : 5.6.1
peft         : 0.18.1
trl          : 1.2.0
bitsandbytes : 0.49.2


In [3]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen2.5-0.5B-Instruct"   # base model from HuggingFace Hub
DATA_FILE      = "train.jsonl"                   # your uploaded JSONL file
OUTPUT_DIR     = "qwen_adapters"                 # where LoRA adapters are saved
MAX_SEQ_LEN    = 2048

# LoRA hyperparameters
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05

# Training hyperparameters
BATCH_SIZE     = 2
GRAD_ACCUM     = 4       # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
MAX_STEPS      = 60      # increase for larger datasets (e.g. 200-500)
LEARNING_RATE  = 2e-4
WARMUP_STEPS   = 5
LOGGING_STEPS  = 5
SAVE_STEPS     = 50

print("Configuration loaded.")

Configuration loaded.


In [4]:
# ── Cell 4: Load model in 4-bit (QLoRA) ───────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,      # saves ~0.4 bits per param extra
)

print(f"Loading tokenizer from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token   # Qwen has no separate pad token
tokenizer.padding_side = "right"            # required for causal LM training

print(f"Loading model in 4-bit ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False              # disable KV-cache during training
model.config.pretraining_tp = 1

print(f"Model loaded. Parameters: {model.num_parameters():,}")

Loading tokenizer from Qwen/Qwen2.5-0.5B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading model in 4-bit ...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. Parameters: 494,032,768


In [5]:
# ── Cell 5: Apply LoRA adapters ────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA (enables gradient checkpointing, casts norms to fp32)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1-2% of total params trainable — this is correct for LoRA

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [6]:
# ── Cell 6: Load and format dataset ───────────────────────────────────────────
from datasets import load_dataset

SYSTEM_PROMPT = "You are a helpful materials science assistant."

def format_example(example):
    """
    Formats a single example into Qwen's ChatML format.
    Expected JSONL fields: instruction, input (optional), output
    """
    instruction = example.get("instruction", "").strip()
    user_input  = example.get("input", "").strip()
    output      = example.get("output", "").strip()

    user_content = f"{instruction}\n{user_input}" if user_input else instruction

    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
        f"<|im_start|>assistant\n{output}<|im_end|>"
    )
    return {"text": text}

print(f"Loading dataset from {DATA_FILE} ...")
dataset = load_dataset("json", data_files=DATA_FILE, split="train")
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print(f"Dataset size: {len(dataset)} examples")
print("\nSample formatted prompt:")
print(dataset[0]["text"][:500], "...")

Loading dataset from train.jsonl ...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/6096 [00:00<?, ? examples/s]

Dataset size: 6096 examples

Sample formatted prompt:
<|im_start|>system
You are a helpful materials science assistant.<|im_end|>
<|im_start|>user
Provide a clear definition, an example, and 3 key points for the given question. Format the response with exact headings.
Define ferrite.<|im_end|>
<|im_start|>assistant
Definition:
Ferrite is a foundational concept in materials science that explains how materials are structured and how they behave under various environments.

Example:
Engineers analyze Ferrite to determine the suitability of a material  ...


In [12]:
# ── Cell 7: Training ───────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

use_bf16 = torch.cuda.is_bf16_supported()

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
    max_length=MAX_SEQ_LEN,       # renamed from max_seq_length in trl 0.24
    packing=False,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print("Starting training ...")
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Total steps   : {trainer_stats.global_step}")
print(f"Training loss : {trainer_stats.training_loss:.4f}")
print(f"Runtime       : {trainer_stats.metrics['train_runtime']:.1f}s")

Adding EOS to train dataset:   0%|          | 0/6096 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6096 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training ...


Step,Training Loss
5,2.933900
10,2.607803
15,2.465600
20,2.287577
25,2.095803
30,2.128909
35,2.216555
40,1.948643
45,1.983383
50,2.057088



Training complete!
Total steps   : 60
Training loss : 2.2229
Runtime       : 504.9s


In [13]:
# ── Cell 8: Save adapters & download ──────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapters saved to '{OUTPUT_DIR}/'")

!zip -r qwen_adapters.zip {OUTPUT_DIR}/
print("Zipped to qwen_adapters.zip — download from the file explorer on the left.")

Adapters saved to 'qwen_adapters/'
updating: qwen_adapters/ (stored 0%)
updating: qwen_adapters/README.md (deflated 44%)
updating: qwen_adapters/tokenizer.json (deflated 81%)
updating: qwen_adapters/chat_template.jinja (deflated 71%)
updating: qwen_adapters/tokenizer_config.json (deflated 60%)
updating: qwen_adapters/adapter_model.safetensors (deflated 21%)
updating: qwen_adapters/adapter_config.json (deflated 58%)
  adding: qwen_adapters/checkpoint-60/ (stored 0%)
  adding: qwen_adapters/checkpoint-60/trainer_state.json (deflated 72%)
  adding: qwen_adapters/checkpoint-60/scheduler.pt (deflated 62%)
  adding: qwen_adapters/checkpoint-60/optimizer.pt (deflated 10%)
  adding: qwen_adapters/checkpoint-60/training_args.bin (deflated 53%)
  adding: qwen_adapters/checkpoint-60/README.md (deflated 65%)
  adding: qwen_adapters/checkpoint-60/tokenizer.json (deflated 81%)
  adding: qwen_adapters/checkpoint-60/chat_template.jinja (deflated 71%)
  adding: qwen_adapters/checkpoint-60/tokenizer_con

In [14]:
# ── Cell 9 (Optional): Quick inference test ────────────────────────────────────
from peft import PeftModel
from transformers import pipeline

# Re-enable cache for inference
model.config.use_cache = True
model.eval()

TEST_INSTRUCTION = "What is the Young's modulus of steel?"

prompt = (
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    f"<|im_start|>user\n{TEST_INSTRUCTION}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"Prompt   : {TEST_INSTRUCTION}")
print(f"Response : {response}")

Prompt   : What is the Young's modulus of steel?
Response : Steel has a very high Young's modulus, which makes it one of the strongest and most durable materials known to man. It is also highly resistant to deformation, corrosion, and fracture under stress.

The formula for calculating Young's modulus (E) in steel is:

E = σ / A

Where:
σ = stress,
A = cross-sectional area,
and
ε = strain.

For example, if you have a piece of steel that measures 10 mm x 10 mm, with an applied force of 200 N, then the calculated value of Young's modulus would be approximately 65 GPa.

It's worth noting that this is just one aspect of steel properties that we can observe when studying this material. There are many more factors involved in determining how these properties manifest in engineering applications or industrial processes.


### What to do next
1. Download `qwen_adapters.zip` from the file explorer
2. Extract into your local project directory
3. Load locally with:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = PeftModel.from_pretrained(base, "./qwen_adapters")
model = model.merge_and_unload()  # merges LoRA weights into base model
```